# Build injected dirty tables from BART changes

Reconstructs `datasets/{table}/injected/{type}_r{rate}.csv` by applying each
`bart/*_changes.csv` log to the table's `clean.csv` (see `inject.py`).

A changes file logs several rows for the same cell when one cell is implicated by
several FD constraints; we collapse those to one value per cell (last wins). This
lowers the realized error rate below the nominal label — see `bart/README.md` for
why we drop duplicates instead of using BART's unique-changes mode, and why the
sweep stays a valid runtime-vs-error-rate benchmark.

The per-file drop counts are written to `bart/dedup_report.md` on every run (the
explanation in `bart/README.md` is static). The EGTask config folders under `bart/`
are never touched.

Adjust `TABLES` / `E1_RATES` below to change what gets built.

In [1]:
import sys
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from inject import apply_changes, parse_changes, read_str

CHANGES = ROOT / "bart"
DATASETS = ROOT / "datasets"

# changes-file prefix -> dataset folder
TABLES = {
    "hospital": "hospital_170k",
    "insurance_claims": "insurance_claims_58k",
}
E1_RATES = (5, 10, 15, 20, 25, 30)

# (changes-file stem suffix, output stem)
JOBS = [(f"e1_{r}", f"e1_r{r:02d}") for r in E1_RATES] + [("e2_10", "e2_r10")]

In [2]:
# Build every injected table and collect dedup stats.
stats = []
for prefix, table in TABLES.items():
    clean = read_str(DATASETS / table / "clean.csv")
    out_dir = DATASETS / table / "injected"
    out_dir.mkdir(exist_ok=True)

    for src, dst in JOBS:
        changes = parse_changes(CHANGES / f"{prefix}_{src}_changes.csv")
        deduped = changes.unique(["__row", "attr"], keep="last")
        total, unique = changes.height, deduped.height

        dirty = apply_changes(clean, deduped)
        dirty.write_csv(out_dir / f"{dst}.csv")

        stats.append(
            {
                "file": f"{prefix}_{src}_changes.csv",
                "table": table,
                "change_rows": total,
                "applied_cells": unique,
                "dropped": total - unique,
                "pct_dropped": round((total - unique) / total * 100, 2) if total else 0.0,
            }
        )
        print(f"{table}/injected/{dst}.csv  ({unique} cells, {total - unique} dropped)")

stats = pl.DataFrame(stats)
stats

hospital_170k/injected/e1_r05.csv  (24307 cells, 12140 dropped)
hospital_170k/injected/e1_r10.csv  (48327 cells, 24141 dropped)
hospital_170k/injected/e1_r15.csv  (73809 cells, 33238 dropped)
hospital_170k/injected/e1_r20.csv  (92408 cells, 40503 dropped)
hospital_170k/injected/e1_r25.csv  (117510 cells, 58747 dropped)
hospital_170k/injected/e1_r30.csv  (137056 cells, 73378 dropped)
hospital_170k/injected/e2_r10.csv  (138027 cells, 0 dropped)
insurance_claims_58k/injected/e1_r05.csv  (15696 cells, 10899 dropped)
insurance_claims_58k/injected/e1_r10.csv  (30872 cells, 21858 dropped)
insurance_claims_58k/injected/e1_r15.csv  (45216 cells, 32979 dropped)
insurance_claims_58k/injected/e1_r20.csv  (59824 cells, 43774 dropped)
insurance_claims_58k/injected/e1_r25.csv  (73713 cells, 55347 dropped)
insurance_claims_58k/injected/e1_r30.csv  (87650 cells, 67287 dropped)
insurance_claims_58k/injected/e2_r10.csv  (87973 cells, 0 dropped)


file,table,change_rows,applied_cells,dropped,pct_dropped
str,str,i64,i64,i64,f64
"""hospital_e1_5_changes.csv""","""hospital_170k""",36447,24307,12140,33.31
"""hospital_e1_10_changes.csv""","""hospital_170k""",72468,48327,24141,33.31
"""hospital_e1_15_changes.csv""","""hospital_170k""",107047,73809,33238,31.05
"""hospital_e1_20_changes.csv""","""hospital_170k""",132911,92408,40503,30.47
"""hospital_e1_25_changes.csv""","""hospital_170k""",176257,117510,58747,33.33
…,…,…,…,…,…
"""insurance_claims_e1_15_changes…","""insurance_claims_58k""",78195,45216,32979,42.18
"""insurance_claims_e1_20_changes…","""insurance_claims_58k""",103598,59824,43774,42.25
"""insurance_claims_e1_25_changes…","""insurance_claims_58k""",129060,73713,55347,42.88


In [3]:
# Regenerate the output-only report (the static explanation lives in README.md).
lines = [
    "# Duplicate cells dropped",
    "",
    "Generated by `notebooks/build_injected.ipynb` — see `README.md` for what this means.",
    "",
    "`dropped` is how many change-rows were duplicate cells; `pct_dropped` is that as a",
    "share of all change-rows. `applied_cells` is the realized number of dirty cells.",
    "",
    "| file | change_rows | applied_cells | dropped | pct_dropped |",
    "|------|------------:|--------------:|--------:|------------:|",
]
for r in stats.iter_rows(named=True):
    lines.append(
        f"| {r['file']} | {r['change_rows']} | {r['applied_cells']} | "
        f"{r['dropped']} | {r['pct_dropped']:.2f}% |"
    )

(CHANGES / "dedup_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {CHANGES / 'dedup_report.md'}")

wrote c:\Users\dvojk\repositories\horizon_project\bart\dedup_report.md
